# Legacy Hour-Aggregated Ridership Analysis

This notebook uses the older hour-aggregated ridership files. It is useful as a comparison to the original analysis, but the preferred DiDisc workflow now uses the raw cleaned ride-event data and panels in `data/processed/Analysis_Ready/`.

## Setup and Aggregated Data Load

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from pathlib import Path


def find_project_root(start=None):
    start = (Path.cwd() if start is None else Path(start)).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() and (candidate / "scripts").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing data/ and scripts/.")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"


In [ ]:
BASE_DIR = RAW_DIR / "TCAT_Ridership_Aggregated"

In [ ]:
AGGREGATED_ANALYSIS_DIR = PROCESSED_DIR / "Aggregated_Analysis"


def _read_any_table(path: Path, **read_kwargs) -> pd.DataFrame:
    suffix = path.suffix.lower()

    if suffix in [".csv", ".txt"]:
        return pd.read_csv(path, header=1, **read_kwargs)

    if suffix in [".xlsx", ".xls"]:
        return pd.read_excel(path, header=1, **read_kwargs)

    raise ValueError(f"Unsupported file type: {path}")


def import_all_routes_from_raw(base_dir: Path = BASE_DIR):
    # Read the original route-level files and concatenate the hour/trip tables.
    by_hour = {}
    by_trip = {}

    for route_dir in sorted([p for p in base_dir.iterdir() if p.is_dir()]):
        route = route_dir.name
        hour_candidates = list(route_dir.glob(f"{route} APC by Hour.*"))
        trip_candidates = list(route_dir.glob(f"{route} APC by Trip.*"))

        if len(hour_candidates) == 0:
            print(f"[WARN] Missing by-hour file for route {route} in {route_dir}")
        elif len(hour_candidates) > 1:
            print(f"[WARN] Multiple by-hour files for route {route}: {[p.name for p in hour_candidates]} (using first)")

        if len(trip_candidates) == 0:
            print(f"[WARN] Missing by-trip file for route {route} in {route_dir}")
        elif len(trip_candidates) > 1:
            print(f"[WARN] Multiple by-trip files for route {route}: {[p.name for p in trip_candidates]} (using first)")

        if hour_candidates:
            df_h = _read_any_table(hour_candidates[0])
            df_h["route"] = route
            by_hour[route] = df_h

        if trip_candidates:
            df_t = _read_any_table(trip_candidates[0])
            df_t["route"] = route
            by_trip[route] = df_t

    by_hour_all = pd.concat(by_hour.values(), ignore_index=True) if by_hour else pd.DataFrame()
    by_trip_all = pd.concat(by_trip.values(), ignore_index=True) if by_trip else pd.DataFrame()
    return by_hour, by_trip, by_hour_all, by_trip_all


def split_by_route(frame: pd.DataFrame) -> dict[str, pd.DataFrame]:
    if frame.empty or "route" not in frame:
        return {}
    return {str(route): group.copy() for route, group in frame.groupby("route", dropna=False)}


def import_all_routes(prefer_parquet: bool = True):
    hour_path = AGGREGATED_ANALYSIS_DIR / "apc_by_hour.parquet"
    trip_path = AGGREGATED_ANALYSIS_DIR / "apc_by_trip.parquet"

    if prefer_parquet and hour_path.exists() and trip_path.exists():
        by_hour_all = pd.read_parquet(hour_path)
        by_trip_all = pd.read_parquet(trip_path)
        by_hour = split_by_route(by_hour_all)
        by_trip = split_by_route(by_trip_all)
        print(f"Loaded concatenated Parquet tables from {AGGREGATED_ANALYSIS_DIR}")
        return by_hour, by_trip, by_hour_all, by_trip_all

    print("Parquet tables not found; reading the original aggregated Excel files.")
    return import_all_routes_from_raw()


by_hour_dict, by_trip_dict, by_hour_all, by_trip_all = import_all_routes()
print(by_hour_all.shape, by_trip_all.shape)


In [ ]:
by_hour_all

In [ ]:
by_hour_all.dtypes

In [ ]:
by_hour_all = by_hour_all[by_hour_all["Hour"] != "Total"].copy()

by_hour_all['APC Boards'] = pd.to_numeric(by_hour_all['APC Boards'], errors='coerce')
by_hour_all['APC Alights'] = pd.to_numeric(by_hour_all['APC Alights'], errors='coerce')
by_hour_all['Hour'] = pd.to_numeric(by_hour_all['Hour'], errors='coerce')

In [ ]:
by_hour_all["APC Boards Standardized"] = (
    by_hour_all
        .groupby("route")["APC Boards"]
        .transform(lambda x: (x - x.mean()) / x.std())
)

## Legacy Aggregated DiDisc Models

These cells estimate preliminary discontinuity models on hourly aggregate APC boards. Treat this section as a baseline check; the cleaner design should use route or route-stop exposure labels from the raw-data pipeline.

In [ ]:
# Use raw boards
rd = by_hour_all.copy()

In [ ]:
cornell_routes = [
    "10","20","30","30W","31","31W","32","37",
    "40","43","51","52",
    "81","82","90","92"
]

cornell_only = ["30","51","82"]

no_cornell = ["11","13","14","14S","15","17","21","36","65","67"]


In [ ]:
rd["running"] = rd["Hour"] - 18
rd["post"] = (rd["running"] >= 0).astype(int)

rd = rd[(rd["running"] <= 2) & (rd["running"] >= -3)]

In [ ]:
rd_subset = rd[
    rd["route"].isin(cornell_only + no_cornell)
].copy()

rd_subset["treated"] = rd_subset["route"].isin(cornell_only).astype(int)

model_pure = smf.ols(
    "Q('APC Boards') ~ post + running + post:running + post:treated",
    data=rd_subset
).fit(cov_type="cluster", cov_kwds={"groups": rd_subset["route"]})

print(model_pure.summary())

In [ ]:
rd_subset = rd_subset[rd_subset["APC Boards"] > 0].copy()
rd_subset["log_boards"] = np.log(rd_subset["APC Boards"])

model_pct = smf.ols(
    "log_boards ~ post + running + post:running + post:treated",
    data=rd_subset
).fit(cov_type="cluster", cov_kwds={"groups": rd_subset["route"]})

print(model_pct.summary())

In [ ]:
rd_subset = rd_subset[rd_subset["APC Boards"] > 0].copy()
rd_subset["log_boards"] = np.log(rd_subset["APC Boards"])

model_pct = smf.ols(
    "log_boards ~ post + running + post:running + post:treated + C(route)",
    data=rd_subset
).fit(cov_type="cluster", cov_kwds={"groups": rd_subset["route"]})

print(model_pct.summary())

In [ ]:
# Use raw boards
rd_placebo = by_hour_all.copy()

rd_placebo["running"] = rd_placebo["Hour"] - 11
rd_placebo["post"] = (rd_placebo["running"] >= 0).astype(int)

rd_placebo = rd_placebo[(rd_placebo["running"] <= 2) & (rd_placebo["running"] >= -3)]

rd_subset = rd_placebo[
    rd_placebo["route"].isin(cornell_only + no_cornell)
].copy()

rd_subset["treated"] = rd_subset["route"].isin(cornell_only).astype(int)

rd_subset = rd_subset[rd_subset["APC Boards"] > 0].copy()
rd_subset["log_boards"] = np.log(rd_subset["APC Boards"])

model_pct = smf.ols(
    "log_boards ~ post + running + post:running + post:treated + C(route)",
    data=rd_subset
).fit(cov_type="cluster", cov_kwds={"groups": rd_subset["route"]})

print(model_pct.summary())

## Legacy Plots

In [ ]:
plot_data = rd_subset.copy()

# Aggregate mean ridership by hour and treatment group
plot_data = (
    plot_data
    .groupby(["Hour","treated"])["APC Boards"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(8,5))

sns.scatterplot(
    data=plot_data,
    x="Hour",
    y="APC Boards",
    hue="treated",
    s=80
)

sns.lineplot(
    data=plot_data,
    x="Hour",
    y="APC Boards",
    hue="treated",
    legend=False
)

plt.axvline(17.5, color="black", linestyle="--")

plt.xlabel("Hour of day")
plt.ylabel("Average Ridership")
plt.title("Ridership Around 18:00 Cutoff\nCornell-Dominant vs Non-Cornell Routes")

plt.show()

In [ ]:
plot_data = rd_subset.copy()

# Aggregate mean ridership by hour and treatment group
plot_data = (
    plot_data
    .groupby(["Hour","treated"])["APC Boards"]
    .mean()
    .reset_index()
)

# Use hour 17 as baseline for each group
baseline = plot_data[plot_data["Hour"] == 17][["treated","APC Boards"]]
baseline = baseline.rename(columns={"APC Boards":"baseline"})

plot_data = plot_data.merge(baseline, on="treated")

# Percentage change relative to hour 17
plot_data["pct_change"] = (
    (plot_data["APC Boards"] - plot_data["baseline"])
    / plot_data["baseline"] * 100
)

plt.figure(figsize=(8,5))

sns.scatterplot(
    data=plot_data,
    x="Hour",
    y="pct_change",
    hue="treated",
    s=80
)

sns.lineplot(
    data=plot_data,
    x="Hour",
    y="pct_change",
    hue="treated",
    legend=False
)

plt.axvline(17.5, color="black", linestyle="--")

plt.xlabel("Hour of day")
plt.ylabel("Percent Change from 17:00 (%)")
plt.title("Percent Change in Ridership Around 18:00 Cutoff\nCornell-Dominant vs Non-Cornell Routes")

plt.axhline(0, color="gray", linestyle="--")

plt.show()

In [ ]:
plot_data = rd_subset.copy()

plot_data = (
    plot_data
    .groupby(["Hour","treated"])["log_boards"]
    .mean()
    .reset_index()
)

# Map treatment labels for clarity
plot_data["group"] = plot_data["treated"].map({
    0: "Non-Cornell Routes",
    1: "Cornell-Dominant Routes"
})

plt.figure(figsize=(8,5))

sns.scatterplot(
    data=plot_data,
    x="Hour",
    y="log_boards",
    hue="group",
    s=90
)

sns.lineplot(
    data=plot_data,
    x="Hour",
    y="log_boards",
    hue="group",
    legend=False
)

# Policy cutoff
plt.axvline(17.5, color="black", linestyle="--", label="Free-Ride Policy Begins")

plt.xlabel("Hour of Day")
plt.ylabel("Log of Hourly Ridership (Boardings)")
plt.title("Ridership Around the 18:00 Cornell Unlimited Access Cutoff")

plt.legend(title="")

plt.tight_layout()
plt.show()

## Bridge to Current Analysis-Ready Panels

Use this section to compare the old hourly aggregate approach with the current cleaned route panel. It does not replace the main raw-data notebook; it just points the old analysis toward the current canonical output.

In [ ]:
from pathlib import Path

analysis_ready_dir = PROCESSED_DIR / "Analysis_Ready"
route_panel_path = analysis_ready_dir / "route_15min_panel.parquet"

if route_panel_path.exists():
    route_panel = pd.read_parquet(route_panel_path)
    current_hourly = (
        route_panel.assign(Hour=route_panel["time_bin_15min"].dt.hour)
        .groupby(["route_clean", "route_group_primary", "Hour"], as_index=False)
        .agg(boards=("boards", "sum"), intervals=("time_bin_15min", "nunique"))
        .sort_values(["route_group_primary", "route_clean", "Hour"])
    )
    display(current_hourly.head(30))
else:
    print("Run scripts/build_analysis_ready_ridership.py to create the current route panel.")
